# Phase 6 - LangGraph Workflow

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [10]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [4]:
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import sqlite3
import threading
import statistics
from datetime import datetime, timezone

ROOT = Path("/home/sagemaker-user/complaintiq")
os.chdir(ROOT)

SRC = ROOT / "src"
STATE = ROOT / "state"
REPORTS = ROOT / "reports"

for path in [SRC, STATE, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project initialized:", ROOT)

Project initialized: /home/sagemaker-user/complaintiq


In [24]:
import json
from pathlib import Path

ROOT = Path("/home/sagemaker-user/complaintiq")

NOTEBOOK4_PATH = ROOT / "notebooks" / "04_agents.ipynb"
NOTEBOOK3_PATH = ROOT / "notebooks" / "03_retrieval.ipynb"

with NOTEBOOK4_PATH.open("r", encoding="utf-8") as f:
    nb4 = json.load(f)

with NOTEBOOK3_PATH.open("r", encoding="utf-8") as f:
    nb3 = json.load(f)

print("Notebook 4 loaded:", NOTEBOOK4_PATH.exists())
print("Notebook 3 loaded:", NOTEBOOK3_PATH.exists())
print("Notebook 4 cells:", len(nb4["cells"]))
print("Notebook 3 cells:", len(nb3["cells"]))

Notebook 4 loaded: True
Notebook 3 loaded: True
Notebook 4 cells: 108
Notebook 3 cells: 54


In [5]:
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import sqlite3
import threading
import statistics
from datetime import datetime, timezone
from src.config import (
    ROOT,
    RAW,
    CFPB_RAW,
    ECFR_RAW,
    PDF_RAW,
    PROC,
    UA,
    INDEX
)
ROOT = Path("/home/sagemaker-user/complaintiq")
os.chdir(ROOT)

SRC = ROOT / "src"
STATE = ROOT / "state"
REPORTS = ROOT / "reports"

for path in [SRC, STATE, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Source directory:", SRC)
print("State directory:", STATE)
print("Reports directory:", REPORTS)

Project root: /home/sagemaker-user/complaintiq
Source directory: /home/sagemaker-user/complaintiq/src
State directory: /home/sagemaker-user/complaintiq/state
Reports directory: /home/sagemaker-user/complaintiq/reports


In [6]:
from config import REGION, STATE as CONFIG_STATE

MODELS_PATH = ROOT / "models.json"

assert REGION == "us-east-1", f"Unexpected region: {REGION}"
assert MODELS_PATH.exists(), "models.json is missing"

MODELS = json.loads(MODELS_PATH.read_text())

print("Region:", REGION)
print("Resolved model roles:")

for role, model_id in MODELS.items():
    print(f"{role:10s}: {model_id}")

assert MODELS.get("worker"), "Worker model is not configured"
assert MODELS.get("judge"), "Judge model is not configured"

print("\nPASS — configuration and model IDs loaded")

Region: us-east-1
Resolved model roles:
worker    : us.anthropic.claude-sonnet-4-6
judge     : us.anthropic.claude-opus-4-8
fast      : us.anthropic.claude-sonnet-4-6
embed     : amazon.titan-embed-text-v2:0

PASS — configuration and model IDs loaded


In [8]:
import importlib
import llm

importlib.reload(llm)

from llm import (
    chat,
    chat_json,
    LEDGER,
    PRICE_PER_1K,
    statistics,
    configure_price,
    cache_stats,
    get_usage_report
)

print("Bedrock wrapper:", llm.__file__)
print("Region:", llm.REGION)
print("Cache:", llm.CACHE_PATH)
print("price:", llm.configure_price)

assert llm.REGION == "us-east-1"

print("PASS — Bedrock wrapper restored")

Bedrock wrapper: /home/sagemaker-user/complaintiq/src/llm.py
Region: us-east-1
Cache: /home/sagemaker-user/complaintiq/state/llm_cache.sqlite
price: <function configure_price at 0x7fc77d968e00>
PASS — Bedrock wrapper restored


In [14]:
import boto3

session = boto3.Session(region_name="us-east-1")
identity = session.client("sts").get_caller_identity()

print("AWS Account:", identity["Account"])
print("AWS ARN:", identity["Arn"])
print("AWS Region:", session.region_name)

print("\nBedrock client region:", llm.brt.meta.region_name)

assert llm.brt.meta.region_name == "us-east-1"

print("\nPASS — Bedrock client is configured for us-east-1")

AWS Account: 456427397380
AWS ARN: arn:aws:sts::456427397380:assumed-role/AmazonSageMaker-ExecutionRole-20261106T002068/SageMaker
AWS Region: us-east-1

Bedrock client region: us-east-1

PASS — Bedrock client is configured for us-east-1


In [12]:
import boto3

REGION = "us-east-1"
BUCKET = "sagemaker-complaintiq-456427397380"

s3 = boto3.client("s3", region_name=REGION)

print("AWS Region:", REGION)
print("S3 Bucket:", BUCKET)

# Verify bucket access
s3.head_bucket(Bucket=BUCKET)

print("PASS — S3 bucket is accessible.")

AWS Region: us-east-1
S3 Bucket: sagemaker-complaintiq-456427397380
PASS — S3 bucket is accessible.


In [16]:
from pathlib import Path
import json
import sqlite3
import operator
import importlib.metadata as metadata

print("=== COMPLAINTIQ RUNTIME VALIDATION ===")

# --------------------------------------------------
# 1. Project configuration
# --------------------------------------------------

print("\nProject configuration:")

print("ROOT:", ROOT)
print("STATE:", STATE)
print("BUCKET:", BUCKET)
print("REGION:", REGION)

assert Path(ROOT).exists(), f"Project root not found: {ROOT}"
assert Path(STATE).exists(), f"State directory not found: {STATE}"
assert BUCKET, "S3 bucket variable is empty"
assert REGION == "us-east-1", f"Unexpected region: {REGION}"

# --------------------------------------------------
# 2. Required runtime objects
# --------------------------------------------------

print("\nRequired runtime objects:")

assert "s3" in globals(), (
    "S3 client missing. Run the existing S3 client initialization cell."
)

assert "MODELS" in globals(), (
    "MODELS missing. Load the existing models.json configuration."
)

assert "llm" in globals(), (
    "llm module missing. Run the existing LLM import cell."
)

# --------------------------------------------------
# 3. Bedrock client
# --------------------------------------------------

assert hasattr(llm, "brt"), (
    "Bedrock client missing inside llm module. "
    "Check src/llm.py initialization."
)

# Expose module client as a notebook-level variable
brt = llm.brt

print("s3:", "AVAILABLE")
print("MODELS:", "AVAILABLE")
print("llm:", "AVAILABLE")
print("brt:", "AVAILABLE")

print("Bedrock service:", brt.meta.service_model.service_name)
print("Bedrock region:", brt.meta.region_name)

assert brt.meta.region_name == REGION, (
    f"Bedrock region mismatch: {brt.meta.region_name}"
)

# --------------------------------------------------
# 4. LangGraph package validation
# --------------------------------------------------

print("\nLangGraph packages:")

for package in ["langgraph", "langgraph-checkpoint-sqlite"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# --------------------------------------------------
# 5. Final result
# --------------------------------------------------

print("\nPASS — ComplaintIQ runtime validation completed.")

=== COMPLAINTIQ RUNTIME VALIDATION ===

Project configuration:
ROOT: /home/sagemaker-user/complaintiq
STATE: /home/sagemaker-user/complaintiq/state
BUCKET: sagemaker-complaintiq-456427397380
REGION: us-east-1

Required runtime objects:
s3: AVAILABLE
MODELS: AVAILABLE
llm: AVAILABLE
brt: AVAILABLE
Bedrock service: bedrock-runtime
Bedrock region: us-east-1

LangGraph packages:
langgraph: 1.2.20
langgraph-checkpoint-sqlite: NOT INSTALLED

PASS — ComplaintIQ runtime validation completed.


In [25]:
print("========== NOTEBOOK 3: INDEX INITIALIZATION ==========")

index_terms = [
    "CASE_INDEX",
    "REG_INDEX",
    "faiss",
    "IndexFlat",
    "read_index",
    "case_vectors.npy",
    "reg_vectors.npy",
    "case_chunks.jsonl",
    "reg_chunks.jsonl",
]

for i, cell in enumerate(nb3["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(term.lower() in source.lower() for term in index_terms):
        print(f"\n--- Notebook 3 | Cell {i} ---")
        print(source[:5000])


print("\n========== NOTEBOOK 4: AGENT DEFINITIONS ==========")

agent_names = [
    "CIQState",
    "guard_node",
    "router_node",
    "triage_node",
    "cases_node",
    "analyst_node",
    "research_node",
    "drafter_node",
    "critic_node",
    "human_node",
    "refuse_node",
]

for i, cell in enumerate(nb4["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    matches = [
        name for name in agent_names
        if f"def {name}" in source or f"class {name}" in source
    ]

    if matches:
        print(f"\n--- Notebook 4 | Cell {i} | {matches} ---")
        print(source[:500])

========== NOTEBOOK 3: INDEX INITIALIZATION ==========

--- Notebook 3 | Cell 6 ---

%pip install -q boto3 faiss-cpu rank-bm25 tqdm


--- Notebook 3 | Cell 15 ---
import os
import json
import hashlib
import boto3
import numpy as np
import pandas as pd
import re
import faiss
import numpy as np
from rank_bm25 import BM25Okapi


from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

print("PASS — All required imports loaded.")

--- Notebook 3 | Cell 17 ---
reg_texts = [r["text"] for r in reg_chunks]

reg_vectors = embed(
    reg_texts,
    workers=3,
    desc="Regulatory/document embeddings"
)

assert reg_vectors.shape == (1784, 1024)
assert np.isfinite(reg_vectors).all()

reg_vectors_path = INDEX / "reg_vectors.npy"

np.save(reg_vectors_path, reg_vectors)

print("\nSaved:", reg_vectors_path)
print("Vector shape:", reg_vectors.shape)
print("Cache files:", len(list(CACHE.glob("*.npy"))))
print("Vector norm sample:", np.linalg.norm(reg_vectors[0]))

In [22]:
from pathlib import Path
import importlib.metadata as metadata

print("=== PROJECT INITIALIZATION ===")

for name in ["ROOT", "BUCKET", "MODELS", "s3", "llm", "brt"]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

if "ROOT" in globals():
    print("Project:", ROOT)

if "BUCKET" in globals():
    print("Bucket:", BUCKET)

if "MODELS" in globals():
    print("Models:", MODELS)

print("\n=== PROJECT FILES ===")

ROOT = Path("/home/sagemaker-user/complaintiq")

for path in [
    ROOT / "data" / "complaintiq.duckdb",
    ROOT / "state",
    ROOT / "index",
    ROOT / "notebooks" / "04_agents.ipynb",
]:
    print(f"{'AVAILABLE' if path.exists() else 'MISSING'}: {path}")

print("\n=== LANGGRAPH PACKAGES ===")

for package in [
    "langgraph",
    "langgraph-checkpoint",
    "langgraph-checkpoint-sqlite",
]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

print("\n=== AGENT FUNCTIONS IN THIS KERNEL ===")

for name in [
    "CIQState", "CASE_INDEX", "REG_INDEX",
    "guard_node", "router_node", "triage_node",
    "cases_node", "analyst_node", "research_node",
    "drafter_node", "critic_node", "human_node",
    "refuse_node", "publish_node",
]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

=== PROJECT INITIALIZATION ===
ROOT: AVAILABLE
BUCKET: AVAILABLE
MODELS: AVAILABLE
s3: AVAILABLE
llm: AVAILABLE
brt: AVAILABLE
Project: /home/sagemaker-user/complaintiq
Bucket: sagemaker-complaintiq-456427397380
Models: {'worker': 'us.anthropic.claude-sonnet-4-6', 'judge': 'us.anthropic.claude-opus-4-8', 'fast': 'us.anthropic.claude-sonnet-4-6', 'embed': 'amazon.titan-embed-text-v2:0'}

=== PROJECT FILES ===
AVAILABLE: /home/sagemaker-user/complaintiq/data/complaintiq.duckdb
AVAILABLE: /home/sagemaker-user/complaintiq/state
AVAILABLE: /home/sagemaker-user/complaintiq/index
AVAILABLE: /home/sagemaker-user/complaintiq/notebooks/04_agents.ipynb

=== LANGGRAPH PACKAGES ===
langgraph 1.2.20
langgraph-checkpoint 4.3.1
langgraph-checkpoint-sqlite 3.1.3

=== AGENT FUNCTIONS IN THIS KERNEL ===
CIQState: MISSING
CASE_INDEX: MISSING
REG_INDEX: MISSING
guard_node: MISSING
router_node: MISSING
triage_node: MISSING
cases_node: MISSING
analyst_node: MISSING
research_node: MISSING
drafter_node: MISSIN

In [23]:
from pathlib import Path

ROOT = Path("/home/sagemaker-user/complaintiq")
INDEX_DIR = ROOT / "index"
STATE_DIR = ROOT / "state"

print("========== INDEX DIRECTORY ==========")
for path in sorted(INDEX_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(ROOT)} | {path.stat().st_size:,} bytes")

print("\n========== STATE DIRECTORY ==========")
for path in sorted(STATE_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(ROOT)} | {path.stat().st_size:,} bytes")

print("\n========== PHASE 3 S3 ARTIFACTS ==========")
response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix="complaintiq/index/"
)

for obj in response.get("Contents", []):
    print(f"{obj['Key']} | {obj['Size']:,} bytes")

========== INDEX DIRECTORY ==========
index/case_chunks.jsonl | 4,897,915 bytes
index/case_retrieval_smoke_tests.jsonl | 3,705 bytes
index/case_vectors.npy | 12,288,128 bytes
index/embed_cache/0011056913cf0ec69391c1c183afc1789e37f17c.npy | 4,224 bytes
index/embed_cache/0017c304719e1cc8f0cc481667d763aea846222c.npy | 4,224 bytes
index/embed_cache/001e4e4a6fe6b8163f55a1c759707431343e69c5.npy | 4,224 bytes
index/embed_cache/003c59c7df8d0becd218331260d0f4dc12f4ad43.npy | 4,224 bytes
index/embed_cache/003cc06b5d7b621547ad0535a0297fa1376a8fc1.npy | 4,224 bytes
index/embed_cache/0045fdc0437ac06f8deb6018ef246f369de4c333.npy | 4,224 bytes
index/embed_cache/0057fd048344b1f6d40c3a283c95fe76a1b39b9a.npy | 4,224 bytes
index/embed_cache/005836820abdd56f3452407dce7b1948dd571a04.npy | 4,224 bytes
index/embed_cache/005a1e112a2975880199bd7aa86b12ff22a82d91.npy | 4,224 bytes
index/embed_cache/00a05a95c4aca09afa8ad4bf540c6e0b4c159cb6.npy | 4,224 bytes
index/embed_cache/00b5514dd7cd0229713e391a9e01f03c3ab3c

In [27]:
import ast
import json
from pathlib import Path

ROOT = Path("/home/sagemaker-user/complaintiq")

nb4_path = ROOT / "notebooks" / "04_agents.ipynb"

with open(nb4_path, "r", encoding="utf-8") as f:
    nb4 = json.load(f)

# Only these existing Notebook 4 implementation cells
selected_cells = [44, 46, 48, 50, 52, 60, 61, 62, 63, 67]

# Required assignments that initialize agent dependencies.
required_assignments = {
    "ROOT", "DB_PATH", "Intent", "PII", "INJECTION",
    "ROUTER_SYS", "con", "TAXONOMY", "ANALYST_SYS",
    "ALLOWED_TABLES", "ro", "SCHEMA_DOC",
    "DRAFT_SYS", "COMMON_DRAFT_RULES", "CRITIC_SYS",
}

loaded = []

for cell_no in selected_cells:
    source = "".join(nb4["cells"][cell_no]["source"])
    tree = ast.parse(source)

    selected_nodes = []

    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom,
                              ast.FunctionDef, ast.AsyncFunctionDef,
                              ast.ClassDef)):
            selected_nodes.append(node)

        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]

            names = {
                target.id
                for target in targets
                if isinstance(target, ast.Name)
            }

            if names & required_assignments:
                selected_nodes.append(node)

    if selected_nodes:
        extracted = ast.Module(body=selected_nodes, type_ignores=[])
        code = compile(extracted, f"notebook4_cell_{cell_no}", "exec")
        exec(code, globals())
        loaded.append(cell_no)

print("Executed implementation cells:", loaded)

required = [
    "CIQState",
    "guard_node",
    "router_node",
    "triage_node",
    "analyst_node",
    "cases_node",
    "research_node",
    "drafter_node",
    "critic_node",
    "human_node",
    "refuse_node",
]

for name in required:
    print(f"{'PASS' if name in globals() else 'MISSING'} | {name}")

Executed implementation cells: [44, 46, 48, 50, 52, 60, 61, 62, 63, 67]
PASS | CIQState
PASS | guard_node
PASS | router_node
PASS | triage_node
PASS | analyst_node
PASS | cases_node
PASS | research_node
PASS | drafter_node
PASS | critic_node
PASS | human_node
PASS | refuse_node


In [28]:
print("CIQState:", "CIQState" in globals())
print("Database:", DB_PATH.exists())
print("DuckDB connection:", con.execute("SELECT 1").fetchone())
print("Read-only connection:", ro.execute("SELECT 1").fetchone())

CIQState: True
Database: True
DuckDB connection: (1,)
Read-only connection: (1,)


In [29]:
import ast

print("========== FIND EMBEDDING FUNCTION IN NOTEBOOK 3 ==========")

for i, cell in enumerate(nb3["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name == "embed":
                print(f"\nFound embed() in Notebook 3, Cell {i}")
                print(ast.get_source_segment(source, node))

print("\n========== LOCAL RETRIEVAL ARTIFACTS ==========")

INDEX = ROOT / "index"

for filename in [
    "reg_vectors.npy",
    "reg_chunks.jsonl",
    "case_vectors.npy",
    "case_chunks.jsonl",
]:
    path = INDEX / filename
    print(
        "PASS" if path.exists() else "MISSING",
        filename,
        f"({path.stat().st_size:,} bytes)" if path.exists() else ""
    )

========== FIND EMBEDDING FUNCTION IN NOTEBOOK 3 ==========

Found embed() in Notebook 3, Cell 11
def embed(texts, workers=3, dim=1024, desc="Embedding"):
    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc
            )
        )

    return np.vstack(vectors).astype("float32")

Found embed() in Notebook 3, Cell 16
def embed(texts, workers=3, dim=1024, desc="Embedding"):

    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:

        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc,
                dynamic_

In [30]:
print("========== NOTEBOOK 3: EMBEDDING IMPLEMENTATION ==========")

for cell_no in [9, 10, 11, 12, 13, 14, 15, 16]:
    cell = nb3["cells"][cell_no]

    if cell["cell_type"] == "code":
        print(f"\n========== CELL {cell_no} ==========")
        print("".join(cell["source"]))

print("\n========== HYBRID INDEX IMPLEMENTATION ==========")

print("".join(nb3["cells"][34]["source"]))

========== NOTEBOOK 3: EMBEDDING IMPLEMENTATION ==========

========== CELL 9 ==========

test_body = json.dumps({
    "inputText": "A consumer reports an electronic funds transfer error.",
    "dimensions": 1024,
    "normalize": True
})

response = brt.invoke_model(
    modelId="amazon.titan-embed-text-v2:0",
    body=test_body,
    contentType="application/json",
    accept="application/json"
)

test_vector = np.asarray(
    json.loads(response["body"].read())["embedding"],
    dtype="float32"
)

print("Vector shape:", test_vector.shape)
print("Vector norm:", np.linalg.norm(test_vector))

assert test_vector.shape == (1024,)
assert np.isfinite(test_vector).all()

print("PASS — Titan Text Embeddings V2 is accessible.")


========== CELL 10 ==========


========== CELL 11 ==========

CACHE = INDEX / "embed_cache"
CACHE.mkdir(parents=True, exist_ok=True)

def cache_path(text, dim=1024):
    key = hashlib.sha1(
        f"{dim}|{text}".encode("utf-8")
    ).hexdigest()
    return CACHE / 

In [31]:
# ============================================================
# COMPLAINTIQ — RESTORE PHASE 3 RETRIEVAL LAYER
# Reuse existing vectors and metadata. No corpus re-embedding.
# ============================================================

import os
import json
import hashlib
import re
import faiss
import numpy as np

from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from rank_bm25 import BM25Okapi

# ------------------------------------------------------------
# 1. Paths and existing embedding cache
# ------------------------------------------------------------

INDEX = ROOT / "index"
CACHE = INDEX / "embed_cache"
CACHE.mkdir(parents=True, exist_ok=True)

required_files = [
    INDEX / "reg_vectors.npy",
    INDEX / "reg_chunks.jsonl",
    INDEX / "case_vectors.npy",
    INDEX / "case_chunks.jsonl",
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Missing retrieval artifact: {path}")

print("PASS — All four existing retrieval artifacts found.")


# ------------------------------------------------------------
# 2. Original query embedding implementation
# ------------------------------------------------------------

def cache_path(text, dim=1024):
    key = hashlib.sha1(
        f"{dim}|{text}".encode("utf-8")
    ).hexdigest()

    return CACHE / f"{key}.npy"


def embed_one(text, dim=1024):
    text = str(text or "").strip() or " "
    path = cache_path(text, dim)

    if path.exists():
        return np.load(path).astype("float32")

    body = json.dumps({
        "inputText": text[:30000],
        "dimensions": dim,
        "normalize": True
    })

    response = brt.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=body,
        contentType="application/json",
        accept="application/json"
    )

    vector = np.asarray(
        json.loads(response["body"].read())["embedding"],
        dtype="float32"
    )

    if vector.shape != (dim,):
        raise ValueError(f"Unexpected vector shape: {vector.shape}")

    if not np.isfinite(vector).all():
        raise ValueError("Embedding contains NaN or Inf")

    temp_path = path.with_suffix(".tmp.npy")
    np.save(temp_path, vector)
    os.replace(temp_path, path)

    return vector


def embed(texts, workers=3, dim=1024, desc="Embedding"):
    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc,
                dynamic_ncols=True
            )
        )

    return np.vstack(vectors).astype("float32")


# ------------------------------------------------------------
# 3. JSONL loader
# ------------------------------------------------------------

def read_jsonl(path):
    records = []

    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"Invalid JSON in {path}, line {line_no}"
                    ) from exc

    return records


# ------------------------------------------------------------
# 4. Original HybridIndex implementation
# ------------------------------------------------------------

def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )


class HybridIndex:

    def __init__(self, chunks, vectors):
        self.chunks = list(chunks)
        self.vectors = np.asarray(vectors, dtype="float32")

        if len(self.vectors) != len(self.chunks):
            raise ValueError("Vector/metadata count mismatch")

        if not np.isfinite(self.vectors).all():
            raise ValueError("Invalid vectors")

        self.faiss = faiss.IndexFlatIP(
            self.vectors.shape[1]
        )

        self.faiss.add(self.vectors)

        self.bm25 = BM25Okapi([
            tok(c.get("text", ""))
            for c in self.chunks
        ])

    def search(self, query, k=8, pool=40, rrf_k=60):

        if not query or not str(query).strip():
            raise ValueError("Query cannot be blank")

        pool = min(pool, len(self.chunks))

        qv = embed([query], workers=1)

        _, dense_ids = self.faiss.search(qv, pool)

        sparse_scores = self.bm25.get_scores(tok(query))

        sparse_ids = np.argsort(
            sparse_scores
        )[::-1][:pool]

        scores = defaultdict(float)

        for ranking in (dense_ids[0], sparse_ids):
            for rank, idx in enumerate(ranking):
                if int(idx) >= 0:
                    scores[int(idx)] += (
                        1.0 / (rrf_k + rank + 1)
                    )

        citation = re.search(
            r"\b(\d{1,2})\s*CFR\s*(\d{3,4}\.\d+)\b",
            query,
            re.I
        )

        if citation:
            target = (
                f"{citation.group(1)} CFR "
                f"{citation.group(2)}"
            ).lower()

            compact = (
                f"{citation.group(1)}cfr"
                f"{citation.group(2)}"
            ).lower()

            for idx, rec in enumerate(self.chunks):

                haystack = " ".join([
                    str(rec.get("chunk_id", "")),
                    str(rec.get("section", "")),
                    str(rec.get("text", ""))
                ]).lower()

                normalized = re.sub(
                    r"\s+", " ", haystack
                )

                compact_haystack = re.sub(
                    r"[^a-z0-9.]", "", haystack
                )

                if (
                    target in normalized
                    or compact in compact_haystack
                ):
                    scores[idx] += 1.0

        top = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                **self.chunks[i],
                "rrf": round(float(scores[i]), 6)
            }
            for i in top
        ]


# ------------------------------------------------------------
# 5. Load existing saved vectors and metadata
# ------------------------------------------------------------

reg_chunks = read_jsonl(INDEX / "reg_chunks.jsonl")
case_chunks = read_jsonl(INDEX / "case_chunks.jsonl")

reg_vectors = np.load(INDEX / "reg_vectors.npy")
case_vectors = np.load(INDEX / "case_vectors.npy")

print("Regulatory metadata:", len(reg_chunks))
print("Regulatory vectors:", reg_vectors.shape)

print("Complaint metadata:", len(case_chunks))
print("Complaint vectors:", case_vectors.shape)


# ------------------------------------------------------------
# 6. Validate saved artifacts
# ------------------------------------------------------------

assert len(reg_chunks) == 1784
assert reg_vectors.shape == (1784, 1024)

assert len(case_chunks) == 3000
assert case_vectors.shape == (3000, 1024)

assert np.isfinite(reg_vectors).all()
assert np.isfinite(case_vectors).all()


# ------------------------------------------------------------
# 7. Reconstruct in-memory hybrid indexes
# ------------------------------------------------------------

REG_INDEX = HybridIndex(
    reg_chunks,
    reg_vectors
)

CASE_INDEX = HybridIndex(
    case_chunks,
    case_vectors
)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("\nPASS — Regulatory index:", REG_INDEX.faiss.ntotal)
print("PASS — Complaint index:", CASE_INDEX.faiss.ntotal)

print("\nPASS — Phase 3 retrieval layer restored.")

PASS — All four existing retrieval artifacts found.
Regulatory metadata: 1784
Regulatory vectors: (1784, 1024)
Complaint metadata: 3000
Complaint vectors: (3000, 1024)

PASS — Regulatory index: 1784
PASS — Complaint index: 3000

PASS — Phase 3 retrieval layer restored.


In [32]:
# Regulatory retrieval test
reg_results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution procedures",
    k=5
)

assert len(reg_results) > 0

print("REGULATORY RETRIEVAL")
for i, result in enumerate(reg_results, 1):
    print(
        i,
        result.get("chunk_id"),
        result.get("rrf"),
        result.get("text", "")[:150].replace("\n", " ")
    )


# Complaint retrieval test
case_results = CASE_INDEX.search(
    "Unauthorized debit card transfer and delayed bank investigation",
    k=5
)

assert len(case_results) > 0

print("\nCOMPLAINT RETRIEVAL")
for i, result in enumerate(case_results, 1):
    print(
        i,
        result.get("complaint_id"),
        result.get("rrf"),
        result.get("issue")
    )

print("\nPASS — Both retrieval indexes are operational.")

Embedding: 100%|██████████| 1/1 [00:00<00:00, 8065.97it/s]


REGULATORY RETRIEVAL
1 exam_manual_v2_2012_page_0700_chunk_0001 1.032266 CFPB Consumer Laws and Regulations EFTA An error includes any of the following: • An unauthorized EFT. • An incorrect EFT to or from the consumer’s ac
2 exam_manual_v2_2012_page_0699_chunk_0002 1.03055 an EFT, for example, when the card is used to draw cash advances directly from a credit line, only the error resolution provisions of Regulation Z wil
3 exam_manual_v2_2012_page_0734_chunk_0001 1.030536 CFPB Examination Checklist EFTA CFPB Manual V.2 (October 2012) Checklist 9 Yes No NA 35. If the financial institution requires written confirmation of
4 12CFR1026_1026.13_chunk_0004 1.027402 ) Promptly reports that the amount or account is in dispute; (ii) Mails or delivers to the consumer (at the same time the report is made) a written no
5 exam_manual_v2_2012_page_0735_chunk_0001 1.027149 Yes No NA CFPB Examination Checklist EFTA 39. If a bill ing error occurred, does the financial institution not impose a charge r

Embedding: 100%|██████████| 1/1 [00:00<00:00,  5.63it/s]


COMPLAINT RETRIEVAL
1 19532488 0.031514 Problem with a lender or other company charging your account
2 19040619 0.029437 Managing an account
3 18669481 0.029274 Managing an account
4 20174024 0.028992 Problem with a lender or other company charging your account
5 21896172 0.028814 Problem with a lender or other company charging your account

PASS — Both retrieval indexes are operational.


In [33]:
def after_analyst(state: CIQState) -> str:
    intent = state.get("intent")

    if intent == "BRIEF":
        return "research"

    if intent == "ANALYTICS":
        return "drafter"

    raise ValueError(f"Unexpected intent after analyst: {intent}")


def after_critic(state: CIQState) -> str:
    critique = state.get("critique", {})
    passed = critique.get("pass", False)
    revisions = state.get("revisions", 0)
    intent = state.get("intent")

    if not passed:
        if revisions < 2:
            return "drafter"
        return "human"

    if intent in {"TRIAGE", "BRIEF"}:
        return "human"

    return "publish"


# Validate the human routing function transferred from Notebook 4
assert callable(after_human)

# Routing tests
assert after_analyst({"intent": "BRIEF"}) == "research"
assert after_analyst({"intent": "ANALYTICS"}) == "drafter"

assert after_critic({
    "intent": "ANALYTICS",
    "critique": {"pass": False},
    "revisions": 1
}) == "drafter"

assert after_critic({
    "intent": "ANALYTICS",
    "critique": {"pass": False},
    "revisions": 2
}) == "human"

assert after_critic({
    "intent": "TRIAGE",
    "critique": {"pass": True},
    "revisions": 0
}) == "human"

assert after_critic({
    "intent": "REG_QA",
    "critique": {"pass": True},
    "revisions": 0
}) == "publish"

assert after_human({"human_decision": {"action": "approve"}}) == "publish"
assert after_human({"human_decision": {"action": "edit"}}) == "publish"
assert after_human({"human_decision": {"action": "reject"}}) == "END"

print("PASS — All graph-routing functions validated.")

PASS — All graph-routing functions validated.


In [34]:
required_names = [
    "BUCKET",
    "s3",
    "MODELS",
    "S3_REPORT_PREFIX",
    "AUDIT_KEY",
    "get_usage_snapshot",
    "append_audit_record",
    "uuid",
    "json",
    "datetime",
    "timezone",
    "ClientError",
]

for name in required_names:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

BUCKET: AVAILABLE
s3: AVAILABLE
MODELS: AVAILABLE
S3_REPORT_PREFIX: MISSING
AUDIT_KEY: MISSING
get_usage_snapshot: MISSING
append_audit_record: MISSING
uuid: MISSING
json: AVAILABLE
datetime: AVAILABLE
timezone: AVAILABLE
ClientError: MISSING


In [35]:
import uuid
import json
from datetime import datetime, timezone
from botocore.exceptions import ClientError

# S3 destinations
S3_REPORT_PREFIX = "complaintiq/reports"
AUDIT_KEY = "complaintiq/reports/audit.jsonl"

# Validate existing project configuration
assert BUCKET == "sagemaker-complaintiq-456427397380"
assert s3 is not None

print("BUCKET:", BUCKET)
print("REPORT PREFIX:", S3_REPORT_PREFIX)
print("AUDIT KEY:", AUDIT_KEY)
print("PASS — Publishing configuration restored.")

BUCKET: sagemaker-complaintiq-456427397380
REPORT PREFIX: complaintiq/reports
AUDIT KEY: complaintiq/reports/audit.jsonl
PASS — Publishing configuration restored.


In [36]:
def get_usage_snapshot():
    """
    Return the current LLM usage report.
    Supports the usage-report interface used by the Phase 4 llm module.
    """
    if hasattr(llm, "get_usage_report"):
        return llm.get_usage_report()

    raise RuntimeError(
        "llm.get_usage_report() is unavailable. "
        "Check the Phase 4 llm module before publishing."
    )


def append_audit_record(record):
    """
    Append one JSON record to the S3 audit log.

    This read-modify-write implementation is suitable for the
    single-user capstone workflow, but is not concurrency-safe.
    """
    try:
        existing = (
            s3.get_object(Bucket=BUCKET, Key=AUDIT_KEY)["Body"]
            .read()
            .decode("utf-8")
        )
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code")

        if code not in {"NoSuchKey", "404", "NotFound"}:
            raise

        existing = ""

    line = json.dumps(record, default=str) + "\n"

    s3.put_object(
        Bucket=BUCKET,
        Key=AUDIT_KEY,
        Body=(existing + line).encode("utf-8"),
        ContentType="application/x-ndjson"
    )

    return f"s3://{BUCKET}/{AUDIT_KEY}"


# Validate the helper functions without writing to S3
assert callable(get_usage_snapshot)
assert callable(append_audit_record)

print("PASS — Usage snapshot and audit functions restored.")

PASS — Usage snapshot and audit functions restored.


In [37]:
usage_snapshot = get_usage_snapshot()

print(json.dumps(usage_snapshot, indent=2, default=str))

print("\nPASS — LLM usage snapshot is accessible.")

{
  "ledger": {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "latency": {
      "calls": 0,
      "average_ms": 0,
      "p50_ms": 0,
      "p95_ms": 0
    },
    "models_used": []
  },
  "cache": {
    "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
    "cached_responses": 9
  },
  "models_without_temperature": []
}

PASS — LLM usage snapshot is accessible.


In [38]:
def publish_node(state, config=None):
    """
    Publish an approved ComplaintIQ report to S3 and append
    an audit record.

    Returns only incremental trace entries because CIQState.trace
    uses the operator.add reducer.
    """

    # 1. Enforce human approval
    decision = state.get("human_decision", {})
    action = decision.get("action")

    if action not in {"approve", "edit"}:
        raise ValueError(
            "Publishing requires a human approve or edit decision."
        )

    draft = state.get("draft", "").strip()

    if not draft:
        raise ValueError("Cannot publish an empty draft.")

    # 2. Resolve thread ID
    configurable = (config or {}).get("configurable", {})
    thread_id = (
        configurable.get("thread_id")
        or state.get("thread_id")
        or f"complaintiq-{uuid.uuid4().hex[:12]}"
    )

    # 3. Build publishing trace
    publishing_event = {
        "node": "publish",
        "status": "publishing",
        "thread_id": thread_id
    }

    prior_trace = list(state.get("trace", []))
    report_trace = prior_trace + [publishing_event]

    # 4. Build Markdown report
    report = (
        "# ComplaintIQ — Approved Report\n\n"
        f"**Thread ID:** `{thread_id}`\n\n"
        f"**Intent:** `{state.get('intent', 'UNKNOWN')}`\n\n"
        f"**Human action:** `{action}`\n\n"
        "---\n\n"
        "## Report\n\n"
        f"{draft}\n\n"
        "---\n\n"
        "## Execution Trace\n\n"
        "```json\n"
        f"{json.dumps(report_trace, indent=2, default=str)}\n"
        "```\n"
    )

    # 5. Upload report to S3
    report_key = f"{S3_REPORT_PREFIX}/{thread_id}.md"

    s3.put_object(
        Bucket=BUCKET,
        Key=report_key,
        Body=report.encode("utf-8"),
        ContentType="text/markdown"
    )

    report_uri = f"s3://{BUCKET}/{report_key}"

    # 6. Capture usage and audit information
    usage = get_usage_snapshot()

    audit_record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "thread_id": thread_id,
        "intent": state.get("intent"),
        "models_used": [
            MODELS.get("worker"),
            MODELS.get("judge")
        ],
        "usage": usage,
        "critic_verdict": state.get("critique", {}).get("verdict"),
        "critic_pass": state.get("critique", {}).get("pass"),
        "human_action": action,
        "report_s3_uri": report_uri
    }

    audit_uri = append_audit_record(audit_record)

    # 7. Return only NEW trace entries
    published_event = {
        "node": "publish",
        "status": "published",
        "report_uri": report_uri
    }

    return {
        "report_uri": report_uri,
        "trace": [
            publishing_event,
            published_event
        ]
    }


print("PASS — publish_node defined.")

PASS — publish_node defined.


In [39]:
assert callable(publish_node)

# Test rejection when there is no human approval
try:
    publish_node({
        "draft": "Test report",
        "human_decision": {"action": "reject"},
        "intent": "TRIAGE",
        "trace": []
    })
    raise AssertionError("Expected publishing to be rejected.")

except ValueError as exc:
    assert "human approve or edit" in str(exc)
    print("PASS — Rejected report cannot be published.")

# Test rejection when the draft is empty
try:
    publish_node({
        "draft": "",
        "human_decision": {"action": "approve"},
        "intent": "TRIAGE",
        "trace": []
    })
    raise AssertionError("Expected empty report to be rejected.")

except ValueError as exc:
    assert "empty draft" in str(exc)
    print("PASS — Empty report cannot be published.")

print("PASS — Publishing safeguards validated.")

PASS — Rejected report cannot be published.
PASS — Empty report cannot be published.
PASS — Publishing safeguards validated.


In [40]:
required_graph_components = [
    "CIQState",
    "guard_node",
    "router_node",
    "triage_node",
    "cases_node",
    "analyst_node",
    "research_node",
    "drafter_node",
    "critic_node",
    "human_node",
    "publish_node",
    "refuse_node",
    "route_intent",
    "after_analyst",
    "after_critic",
    "after_human"
]

missing = [
    name for name in required_graph_components
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing graph components: {missing}"
    )

print("PASS — All required graph components are available.")

print("\nReady to assemble the ComplaintIQ LangGraph workflow.")

PASS — All required graph components are available.

Ready to assemble the ComplaintIQ LangGraph workflow.


In [41]:
import sqlite3
from pathlib import Path

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

# Correct human routing to use LangGraph's END constant
def after_human(state: CIQState):
    action = state.get("human_decision", {}).get("action")

    if action in {"approve", "edit"}:
        return "publish"

    return END


# Validate human routing
assert after_human({"human_decision": {"action": "approve"}}) == "publish"
assert after_human({"human_decision": {"action": "edit"}}) == "publish"
assert after_human({"human_decision": {"action": "reject"}}) == END

print("PASS — Human approval and rejection routing corrected.")

PASS — Human approval and rejection routing corrected.


In [42]:
# Initialize the state graph
g = StateGraph(CIQState)

# Register all graph nodes
nodes = {
    "guard": guard_node,
    "router": router_node,
    "triage": triage_node,
    "cases": cases_node,
    "analyst": analyst_node,
    "research": research_node,
    "drafter": drafter_node,
    "critic": critic_node,
    "human": human_node,
    "publish": publish_node,
    "refuse": refuse_node,
}

for name, fn in nodes.items():
    g.add_node(name, fn)

# Entry and guard
g.add_edge(START, "guard")
g.add_edge("guard", "router")

# Intent routing
g.add_conditional_edges(
    "router",
    route_intent,
    {
        "TRIAGE": "triage",
        "ANALYTICS": "analyst",
        "BRIEF": "analyst",
        "REG_QA": "research",
        "OUT_OF_SCOPE": "refuse",
    }
)

# Triage workflow
g.add_edge("triage", "cases")
g.add_edge("cases", "research")

# Analytics and brief workflows
g.add_conditional_edges(
    "analyst",
    after_analyst,
    {
        "research": "research",
        "drafter": "drafter",
    }
)

# Research and drafting
g.add_edge("research", "drafter")
g.add_edge("drafter", "critic")

# Critic review and revision loop
g.add_conditional_edges(
    "critic",
    after_critic,
    {
        "drafter": "drafter",
        "human": "human",
        "publish": "publish",
    }
)

# Human approval
g.add_conditional_edges(
    "human",
    after_human,
    {
        "publish": "publish",
        END: END,
    }
)

# Terminal nodes
g.add_edge("publish", END)
g.add_edge("refuse", END)

print("PASS — All graph nodes and edges registered.")

PASS — All graph nodes and edges registered.


In [43]:
# Ensure state directory exists
STATE = ROOT / "state"
STATE.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DB = STATE / "checkpoints.sqlite"

# Persistent SQLite checkpoint connection
checkpoint_conn = sqlite3.connect(
    str(CHECKPOINT_DB),
    check_same_thread=False
)

# Compile graph with persistent checkpointing
app = g.compile(
    checkpointer=SqliteSaver(checkpoint_conn)
)

assert app is not None
assert CHECKPOINT_DB.exists()

print("PASS — ComplaintIQ LangGraph compiled.")
print("Checkpoint database:", CHECKPOINT_DB)
print("Checkpoint database size:", CHECKPOINT_DB.stat().st_size, "bytes")

PASS — ComplaintIQ LangGraph compiled.
Checkpoint database: /home/sagemaker-user/complaintiq/state/checkpoints.sqlite
Checkpoint database size: 0 bytes


In [44]:
# Obtain compiled graph representation
graph_view = app.get_graph()

# Display Mermaid topology
mermaid = graph_view.draw_mermaid()

print(mermaid)

# Verify expected nodes
actual_nodes = set(graph_view.nodes.keys())

expected_nodes = {
    "__start__",
    "__end__",
    "guard",
    "router",
    "triage",
    "cases",
    "analyst",
    "research",
    "drafter",
    "critic",
    "human",
    "publish",
    "refuse",
}

missing_nodes = expected_nodes - actual_nodes

if missing_nodes:
    raise RuntimeError(f"Missing graph nodes: {missing_nodes}")

print("\nPASS — All expected graph nodes exist.")
print("Node count:", len(actual_nodes))
print("\nPASS — COMPLAINTIQ GRAPH COMPILATION COMPLETE.")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	guard(guard)
	router(router)
	triage(triage)
	cases(cases)
	analyst(analyst)
	research(research)
	drafter(drafter)
	critic(critic)
	human(human)
	publish(publish)
	refuse(refuse)
	__end__([<p>__end__</p>]):::last
	__start__ --> guard;
	analyst -.-> drafter;
	analyst -.-> research;
	cases --> research;
	critic -.-> drafter;
	critic -.-> human;
	critic -.-> publish;
	drafter --> critic;
	guard --> router;
	human -.-> __end__;
	human -.-> publish;
	research --> drafter;
	router -. &nbsp;ANALYTICS&nbsp; .-> analyst;
	router -. &nbsp;OUT_OF_SCOPE&nbsp; .-> refuse;
	router -. &nbsp;REG_QA&nbsp; .-> research;
	router -. &nbsp;TRIAGE&nbsp; .-> triage;
	triage --> cases;
	publish --> __end__;
	refuse --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


PASS — All expected graph nodes exist.
Node count: 13

PASS — COMPLAINTIQ GRAPH C

In [45]:
# Access the checkpointer attached to the compiled graph
checkpointer = app.checkpointer

assert checkpointer is not None
assert hasattr(checkpointer, "setup")

# Initialize checkpoint tables
checkpointer.setup()

# Verify SQLite schema
cursor = checkpoint_conn.cursor()

tables = cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""").fetchall()

table_names = [row[0] for row in tables]

print("SQLite tables:")
for table in table_names:
    print(" -", table)

assert len(table_names) > 0, "No checkpoint tables were created."

checkpoint_conn.commit()

print("\nDatabase:", CHECKPOINT_DB)
print("Database size:", CHECKPOINT_DB.stat().st_size, "bytes")
print("PASS — SQLite checkpoint schema initialized.")

SQLite tables:
 - checkpoints
 - writes

Database: /home/sagemaker-user/complaintiq/state/checkpoints.sqlite
Database size: 4096 bytes
PASS — SQLite checkpoint schema initialized.


In [46]:
# Router destination validation
router_routes = {
    "TRIAGE": "triage",
    "ANALYTICS": "analyst",
    "BRIEF": "analyst",
    "REG_QA": "research",
    "OUT_OF_SCOPE": "refuse",
}

for intent, expected in router_routes.items():
    actual = intent
    assert actual in router_routes
    assert router_routes[actual] == expected

# Analyst routes
assert after_analyst({"intent": "BRIEF"}) == "research"
assert after_analyst({"intent": "ANALYTICS"}) == "drafter"

# Critic routes
assert after_critic({
    "intent": "REG_QA",
    "critique": {"pass": True},
    "revisions": 0
}) == "publish"

assert after_critic({
    "intent": "TRIAGE",
    "critique": {"pass": True},
    "revisions": 0
}) == "human"

assert after_critic({
    "intent": "ANALYTICS",
    "critique": {"pass": False},
    "revisions": 1
}) == "drafter"

assert after_critic({
    "intent": "ANALYTICS",
    "critique": {"pass": False},
    "revisions": 2
}) == "human"

# Human routes
assert after_human({"human_decision": {"action": "approve"}}) == "publish"
assert after_human({"human_decision": {"action": "edit"}}) == "publish"
assert after_human({"human_decision": {"action": "reject"}}) == END

print("PASS — All routing scenarios validated.")

PASS — All routing scenarios validated.


In [47]:
# Validate graph nodes
expected_nodes = {
    "__start__", "__end__",
    "guard", "router", "triage", "cases",
    "analyst", "research", "drafter",
    "critic", "human", "publish", "refuse"
}

actual_nodes = set(app.get_graph().nodes.keys())

assert expected_nodes.issubset(actual_nodes)

# Validate retrieval dependencies
assert len(REG_INDEX.chunks) == 1784
assert len(CASE_INDEX.chunks) == 3000

# Validate agent and publishing functions
assert callable(guard_node)
assert callable(router_node)
assert callable(triage_node)
assert callable(cases_node)
assert callable(analyst_node)
assert callable(research_node)
assert callable(drafter_node)
assert callable(critic_node)
assert callable(human_node)
assert callable(publish_node)
assert callable(refuse_node)

# Validate checkpoint initialization
assert checkpointer is not None
assert len(table_names) > 0

print("========================================")
print("COMPLAINTIQ PHASE 6 VALIDATION")
print("========================================")
print("Graph nodes:", len(actual_nodes))
print("Regulatory chunks:", len(REG_INDEX.chunks))
print("Complaint chunks:", len(CASE_INDEX.chunks))
print("Checkpoint tables:", len(table_names))
print("Agent functions: PASS")
print("Publishing safeguards: PASS")
print("Routing functions: PASS")
print("----------------------------------------")
print("PASS — PHASE 6 STRUCTURAL ACCEPTANCE")

COMPLAINTIQ PHASE 6 VALIDATION
Graph nodes: 13
Regulatory chunks: 1784
Complaint chunks: 3000
Checkpoint tables: 2
Agent functions: PASS
Publishing safeguards: PASS
Routing functions: PASS
----------------------------------------
PASS — PHASE 6 STRUCTURAL ACCEPTANCE


In [48]:
from langgraph.types import interrupt, Command

def checkpoint_test_node(state: CIQState):
    decision = interrupt({
        "type": "checkpoint_test",
        "message": "Approve the checkpoint resume test?",
        "thread_id": state.get("thread_id")
    })

    return {
        "human_decision": decision,
        "trace": [{
            "node": "checkpoint_test",
            "status": "resumed"
        }]
    }


test_graph = StateGraph(CIQState)

test_graph.add_node("checkpoint_test", checkpoint_test_node)
test_graph.add_edge(START, "checkpoint_test")
test_graph.add_edge("checkpoint_test", END)

checkpoint_test_app = test_graph.compile(
    checkpointer=SqliteSaver(checkpoint_conn)
)

print("PASS — Checkpoint test graph compiled.")

PASS — Checkpoint test graph compiled.


In [49]:
test_thread_id = f"ciq-checkpoint-test-{uuid.uuid4().hex[:10]}"

test_config = {
    "configurable": {
        "thread_id": test_thread_id
    }
}

initial_test_state = {
    "request": "Test checkpoint persistence and resume.",
    "thread_id": test_thread_id,
    "human_decision": {},
    "trace": []
}

interrupt_result = checkpoint_test_app.invoke(
    initial_test_state,
    config=test_config
)

print("Invocation result:")
print(interrupt_result)

assert "__interrupt__" in interrupt_result
assert len(interrupt_result["__interrupt__"]) > 0

saved_snapshot = checkpoint_test_app.get_state(test_config)

assert saved_snapshot is not None
assert saved_snapshot.next == ("checkpoint_test",)

print("\nPASS — Workflow interrupted and checkpoint saved.")
print("Thread ID:", test_thread_id)
print("Pending nodes:", saved_snapshot.next)

Invocation result:
{'request': 'Test checkpoint persistence and resume.', 'human_decision': {}, 'trace': [], '__interrupt__': [Interrupt(value={'type': 'checkpoint_test', 'message': 'Approve the checkpoint resume test?', 'thread_id': None}, id='7fb997f158662ddc796eaf2f56e8da11', response_schema=None)]}

PASS — Workflow interrupted and checkpoint saved.
Thread ID: ciq-checkpoint-test-cf76f83149
Pending nodes: ('checkpoint_test',)


In [50]:
resume_result = checkpoint_test_app.invoke(
    Command(
        resume={
            "action": "approve",
            "note": "Checkpoint resume validation"
        }
    ),
    config=test_config
)

print("Resume result:")
print(resume_result)

final_snapshot = checkpoint_test_app.get_state(test_config)

assert final_snapshot.values["human_decision"]["action"] == "approve"
assert final_snapshot.next == ()

assert any(
    item.get("status") == "resumed"
    for item in final_snapshot.values.get("trace", [])
)

print("\nPASS — Checkpoint resumed successfully.")
print("Human decision:", final_snapshot.values["human_decision"])
print("Final trace:", final_snapshot.values["trace"])

Resume result:
{'request': 'Test checkpoint persistence and resume.', 'human_decision': {'action': 'approve', 'note': 'Checkpoint resume validation'}, 'trace': [{'node': 'checkpoint_test', 'status': 'resumed'}]}

PASS — Checkpoint resumed successfully.
Human decision: {'action': 'approve', 'note': 'Checkpoint resume validation'}
Final trace: [{'node': 'checkpoint_test', 'status': 'resumed'}]


In [51]:
checkpoint_conn.commit()

checkpoint_count = checkpoint_conn.execute(
    "SELECT COUNT(*) FROM checkpoints"
).fetchone()[0]

write_count = checkpoint_conn.execute(
    "SELECT COUNT(*) FROM writes"
).fetchone()[0]

print("Checkpoint records:", checkpoint_count)
print("Write records:", write_count)
print("Database size:", CHECKPOINT_DB.stat().st_size, "bytes")

assert checkpoint_count > 0
assert write_count > 0

print("\nPASS — SQLite checkpoint persistence verified.")

Checkpoint records: 3
Write records: 9
Database size: 4096 bytes

PASS — SQLite checkpoint persistence verified.


In [52]:
import json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/home/sagemaker-user/complaintiq")
STATE = ROOT / "state"
INDEX = ROOT / "index"
PROC = ROOT / "data" / "processed"

runtime_inventory = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(ROOT),
    "region": "us-east-1",
    "bucket": "sagemaker-complaintiq-456427397380",
    "models": {
        "worker": "us.anthropic.claude-sonnet-4-6",
        "judge": "us.anthropic.claude-opus-4-8",
        "fast": "us.anthropic.claude-sonnet-4-6",
        "embed": "amazon.titan-embed-text-v2:0"
    },
    "phase3_artifacts": [
        "index/reg_vectors.npy",
        "index/reg_chunks.jsonl",
        "index/case_vectors.npy",
        "index/case_chunks.jsonl"
    ],
    "phase4_artifacts": [
        "src/llm.py",
        "state/llm_cache.sqlite"
    ],
    "phase5_source": [
        "notebooks/04_agents.ipynb"
    ],
    "phase6_source": [
        "notebooks/05_graph.ipynb"
    ],
    "checkpoint_database": "state/checkpoints.sqlite"
}

inventory_path = ROOT / "runtime_inventory.json"

inventory_path.write_text(
    json.dumps(runtime_inventory, indent=2),
    encoding="utf-8"
)

print(json.dumps(runtime_inventory, indent=2))
print("\nPASS — Runtime inventory saved:", inventory_path)

{
  "timestamp_utc": "2026-11-06T11:59:03.734806+00:00",
  "project_root": "/home/sagemaker-user/complaintiq",
  "region": "us-east-1",
  "bucket": "sagemaker-complaintiq-456427397380",
  "models": {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
  },
  "phase3_artifacts": [
    "index/reg_vectors.npy",
    "index/reg_chunks.jsonl",
    "index/case_vectors.npy",
    "index/case_chunks.jsonl"
  ],
  "phase4_artifacts": [
    "src/llm.py",
    "state/llm_cache.sqlite"
  ],
  "phase5_source": [
    "notebooks/04_agents.ipynb"
  ],
  "phase6_source": [
    "notebooks/05_graph.ipynb"
  ],
  "checkpoint_database": "state/checkpoints.sqlite"
}

PASS — Runtime inventory saved: /home/sagemaker-user/complaintiq/runtime_inventory.json


In [53]:
import importlib
import llm

importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.get_usage_report)

print(json.dumps(
    llm.get_usage_report(),
    indent=2,
    default=str
))

print("PASS — Phase 4 LLM wrapper restored.")

{
  "ledger": {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "latency": {
      "calls": 0,
      "average_ms": 0,
      "p50_ms": 0,
      "p95_ms": 0
    },
    "models_used": []
  },
  "cache": {
    "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
    "cached_responses": 9
  },
  "models_without_temperature": []
}
PASS — Phase 4 LLM wrapper restored.


In [54]:
required_runtime = [
    "ROOT",
    "BUCKET",
    "MODELS",
    "s3",
    "brt",
    "llm",
    "reg_chunks",
    "case_chunks",
    "reg_vectors",
    "case_vectors",
    "REG_INDEX",
    "CASE_INDEX",
    "CIQState",
    "guard_node",
    "router_node",
    "triage_node",
    "cases_node",
    "analyst_node",
    "research_node",
    "drafter_node",
    "critic_node",
    "human_node",
    "refuse_node",
    "after_analyst",
    "after_critic",
    "after_human",
    "publish_node",
    "app",
    "checkpoint_conn"
]

missing = [
    name for name in required_runtime
    if name not in globals()
]

if missing:
    raise RuntimeError(f"Runtime restoration incomplete: {missing}")

assert len(reg_chunks) == 1784
assert len(case_chunks) == 3000
assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

assert CHECKPOINT_DB.exists()
assert app is not None

print("====================================")
print("COMPLAINTIQ RUNTIME VALIDATION")
print("====================================")
print("Phase 3: PASS")
print("Phase 4: PASS")
print("Phase 5: PASS")
print("Phase 6: PASS")
print("Missing variables:", missing)
print("====================================")
print("PASS — Runtime is ready.")

COMPLAINTIQ RUNTIME VALIDATION
Phase 3: PASS
Phase 4: PASS
Phase 5: PASS
Phase 6: PASS
Missing variables: []
PASS — Runtime is ready.
